# RNN dan LSTM — Neural Network untuk Data Berurutan (Sequential Data)

---

## Tujuan Pembelajaran

Setelah menyelesaikan notebook ini, kamu akan mampu:

1. **Memahami data berurutan (sequential data)** — apa itu dan mengapa perlu penanganan khusus
2. **Menjelaskan cara kerja RNN (Recurrent Neural Network)** — neural network yang punya 'memori'
3. **Memahami masalah vanishing gradient** — kelemahan utama RNN biasa
4. **Menjelaskan solusi LSTM (Long Short-Term Memory)** — cara LSTM mengatasi kelemahan RNN
5. **Mempraktikkan prediksi time series** — prediksi gelombang sinus dengan SimpleRNN vs LSTM vs GRU
6. **Mempraktikkan analisis sentimen (sentiment analysis)** — klasifikasi review film IMDB

**Perkiraan waktu**: ~90 menit

---

In [ ]:
!pip install -q tensorflow matplotlib numpy

In [ ]:
import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt

print(f"TensorFlow version: {tf.__version__}")
if tf.config.list_physical_devices('GPU'):
    print("GPU terdeteksi!")
else:
    print("Tidak ada GPU — tetap bisa jalan.")

## Bagian 1: Data Berurutan (Sequential Data)

### Mengapa Neural Network Biasa Tidak Cukup?

Bayangkan kamu membaca kalimat: *'Saya pergi ke **___**'*. Untuk mengisi kata yang hilang, kamu perlu INGAT kata-kata sebelumnya. Neural network biasa (Dense/CNN) tidak punya memori — setiap input diproses independen.

**Data berurutan** adalah data yang urutannya PENTING:
- Teks: 'Saya suka kucing' tidak sama dengan 'Kucing suka saya'
- Harga saham: urutan naik-turun sangat penting
- Musik: nada berikutnya bergantung pada nada sebelumnya
- Cuaca: suhu hari ini dipengaruhi suhu kemarin

**RNN (Recurrent Neural Network)** adalah neural network yang punya 'memori' — ia mengingat informasi dari langkah sebelumnya.

### Cara Kerja RNN

Analogi: Bayangkan kamu menonton film. Setiap adegan, kamu:
1. Melihat adegan baru (input)
2. Mengingat apa yang terjadi sebelumnya (hidden state / memori)
3. Menggabungkan keduanya untuk memahami cerita (output)

RNN bekerja persis seperti ini — setiap langkah waktu (timestep), ia menerima input baru DAN informasi dari langkah sebelumnya.

```
Timestep 1      Timestep 2      Timestep 3
+---------+     +---------+     +---------+
|   RNN   |--h1>|   RNN   |--h2>|   RNN   |--h3--> Output
+---------+     +---------+     +---------+
     |               |               |
  Input 1         Input 2         Input 3
 ('Saya')        ('suka')       ('kucing')
```

**h** = hidden state = 'memori' yang diteruskan ke langkah berikutnya

In [ ]:
# Demo konsep "memori" dalam data berurutan
np.random.seed(42)

# Buat data suhu harian (simulasi)
days = np.arange(1, 31)
temperature = 25 + 5 * np.sin(days * 0.3) + np.random.normal(0, 1.5, 30)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(days, temperature, 'o-', color='#42A5F5', label='Suhu harian', linewidth=2)

# Tunjukkan "prediksi" berdasarkan memori (moving average)
window = 3
ma = np.convolve(temperature, np.ones(window)/window, mode='valid')
ax.plot(days[window-1:], ma, '--', color='#E53935', label=f'Rata-rata {window} hari terakhir (memori)', linewidth=2)

ax.set_xlabel('Hari')
ax.set_ylabel('Suhu (C)')
ax.set_title('Data Berurutan: Suhu Harian — Prediksi Butuh "Memori"', fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Untuk memprediksi suhu BESOK, kita perlu tahu suhu HARI INI dan KEMARIN.")
print("Inilah mengapa kita butuh RNN — neural network yang punya memori!")

## Bagian 2: Praktik RNN — Prediksi Gelombang Sinus

Mari kita mulai dengan contoh sederhana: memprediksi nilai gelombang sinus berikutnya. Ini seperti memprediksi pasang surut air laut — pola berulang yang bisa dipelajari.

In [ ]:
# Buat data gelombang sinus
t = np.linspace(0, 100, 1000)
data = np.sin(t)

# Buat sequences: setiap input = 50 langkah, target = 1 langkah berikutnya
seq_length = 50
X, y = [], []
for i in range(len(data) - seq_length):
    X.append(data[i:i+seq_length])
    y.append(data[i+seq_length])
X = np.array(X).reshape(-1, seq_length, 1)  # (samples, timesteps, features)
y = np.array(y)

# Split train/test
split = int(len(X) * 0.8)
X_train, X_test = X[:split], X[split:]
y_train, y_test = y[:split], y[split:]

print(f"Input shape : {X_train.shape} (samples, timesteps, features)")
print(f"Target shape: {y_train.shape}")
print(f"Training    : {len(X_train)} sequences")
print(f"Testing     : {len(X_test)} sequences")

# Visualisasi satu sequence
plt.figure(figsize=(12, 3))
plt.plot(range(seq_length), X_train[0].flatten(), 'b-', linewidth=2, label='Input (50 langkah)')
plt.plot(seq_length, y_train[0], 'ro', markersize=10, label='Target (langkah ke-51)')
plt.xlabel('Timestep')
plt.ylabel('Nilai')
plt.title('Satu Contoh Sequence: Input -> Target', fontweight='bold')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Model RNN sederhana
model_rnn = tf.keras.Sequential([
    tf.keras.layers.SimpleRNN(32, activation='tanh', input_shape=(seq_length, 1), name='rnn_layer'),
    tf.keras.layers.Dense(1, name='output')
], name='SimpleRNN')

model_rnn.summary()

model_rnn.compile(optimizer='adam', loss='mse')

print("\nTraining SimpleRNN...")
history_rnn = model_rnn.fit(X_train, y_train, epochs=20, batch_size=32,
                            validation_split=0.1, verbose=1)
print("Selesai!")

### Hasil Prediksi RNN

In [ ]:
from sklearn.metrics import mean_squared_error, mean_absolute_error

# Prediksi
y_pred_rnn = model_rnn.predict(X_test, verbose=0).flatten()

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 7))

# Plot prediksi vs aktual
ax1.plot(y_test, label='Aktual', color='#42A5F5', linewidth=2)
ax1.plot(y_pred_rnn, label='Prediksi RNN', color='#E53935', linewidth=2, alpha=0.8)
ax1.set_title('SimpleRNN: Prediksi vs Aktual', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_xlabel('Sample')

# Training loss
ax2.plot(history_rnn.history['loss'], label='Training Loss', linewidth=2)
ax2.plot(history_rnn.history['val_loss'], label='Validation Loss', linewidth=2)
ax2.set_title('Training Loss', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# Hitung error
mse_rnn = mean_squared_error(y_test, y_pred_rnn)
mae_rnn = mean_absolute_error(y_test, y_pred_rnn)
print(f"SimpleRNN — MSE: {mse_rnn:.6f}, MAE: {mae_rnn:.6f}")

## Bagian 3: Masalah RNN dan Solusi LSTM

### Masalah Vanishing Gradient

RNN punya kelemahan besar: **vanishing gradient** — semakin panjang urutan, semakin 'lupa' RNN pada informasi awal.

Analogi: Bayangkan permainan bisik-bisik berantai. Pesan awal: *'Kucing hitam duduk di atas meja'*. Setelah 20 orang, pesan menjadi: *'Kacang hijau masuk ke emas kerja'*. Semakin panjang rantai, semakin hilang informasi awal!

RNN mengalami hal serupa — gradien (sinyal belajar) menjadi sangat kecil saat di-backpropagate melalui banyak timestep.

### Solusi: LSTM (Long Short-Term Memory)

LSTM mengatasi masalah ini dengan menambahkan 'highway' informasi — jalur khusus yang memungkinkan informasi mengalir langsung dari masa lalu ke masa depan tanpa terdistorsi.

Komponen LSTM:
- **Cell State** (c) — 'memori jangka panjang', seperti buku catatan
- **Forget Gate** — 'hapus info yang tidak relevan' dari catatan
- **Input Gate** — 'tulis info baru' ke catatan
- **Output Gate** — 'baca info' dari catatan untuk output

```
         Cell State (memori jangka panjang)
--------[x forget]---[+ input]------------------>
              |           |           |
          Forget       Input       Output
          Gate         Gate        Gate
            |            |           |
         'Hapus'      'Tulis'     'Baca'
         info lama   info baru   untuk output
```

### Perbandingan RNN vs LSTM

| Aspek | SimpleRNN | LSTM |
|-------|-----------|------|
| Memori | Pendek (5-10 langkah) | Panjang (100+ langkah) |
| Parameter | Sedikit | 4x lebih banyak |
| Vanishing gradient | Sering terjadi | Teratasi |
| Kecepatan | Cepat | Lebih lambat |
| Kapan digunakan | Urutan pendek | Urutan panjang, teks, musik |

In [ ]:
# Model LSTM
model_lstm = tf.keras.Sequential([
    tf.keras.layers.LSTM(32, activation='tanh', input_shape=(seq_length, 1), name='lstm_layer'),
    tf.keras.layers.Dense(1, name='output')
], name='LSTM_Model')

model_lstm.summary()

# Bandingkan jumlah parameter
print(f"\nPerbandingan Parameter:")
print(f"   SimpleRNN : {model_rnn.count_params():,} parameter")
print(f"   LSTM      : {model_lstm.count_params():,} parameter")
print(f"   LSTM punya {model_lstm.count_params()/model_rnn.count_params():.1f}x lebih banyak parameter (4 gates!)")

model_lstm.compile(optimizer='adam', loss='mse')

print("\nTraining LSTM...")
history_lstm = model_lstm.fit(X_train, y_train, epochs=20, batch_size=32,
                              validation_split=0.1, verbose=1)
print("Selesai!")

### Perbandingan Hasil: SimpleRNN vs LSTM

In [ ]:
y_pred_lstm = model_lstm.predict(X_test, verbose=0).flatten()

fig, axes = plt.subplots(2, 1, figsize=(14, 7))

# SimpleRNN
axes[0].plot(y_test, label='Aktual', color='#42A5F5', linewidth=2)
axes[0].plot(y_pred_rnn, label='Prediksi', color='#E53935', linewidth=2, alpha=0.8)
mse_rnn_val = mean_squared_error(y_test, y_pred_rnn)
axes[0].set_title(f'SimpleRNN (MSE: {mse_rnn_val:.6f})', fontweight='bold')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# LSTM
axes[1].plot(y_test, label='Aktual', color='#42A5F5', linewidth=2)
axes[1].plot(y_pred_lstm, label='Prediksi', color='#66BB6A', linewidth=2, alpha=0.8)
mse_lstm_val = mean_squared_error(y_test, y_pred_lstm)
axes[1].set_title(f'LSTM (MSE: {mse_lstm_val:.6f})', fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.suptitle('Perbandingan SimpleRNN vs LSTM pada Prediksi Gelombang Sinus', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

print(f"SimpleRNN MSE: {mse_rnn_val:.6f}")
print(f"LSTM MSE     : {mse_lstm_val:.6f}")
if mse_lstm_val < mse_rnn_val:
    print(f"LSTM {mse_rnn_val/mse_lstm_val:.1f}x lebih akurat!")
else:
    print("Pada data sederhana seperti sinus, keduanya cukup baik.")
    print("Perbedaan nyata terlihat pada data yang lebih kompleks (teks, saham, dll).")

## Bagian 4: Praktik LSTM — Analisis Sentimen Film (IMDB)

Sekarang kita gunakan LSTM untuk tugas NLP (Natural Language Processing): menentukan apakah review film bernada **positif** atau **negatif**.

Dataset IMDB berisi 50.000 review film dari website IMDB. Setiap review sudah dilabeli positif atau negatif.

Contoh alur kerja:
- Input: *"This movie was absolutely fantastic, I loved every minute of it!"*
- Output: **POSITIF** (1)

- Input: *"Terrible acting and boring plot, complete waste of time."*
- Output: **NEGATIF** (0)

In [ ]:
# Load dataset IMDB (sudah built-in di TensorFlow)
vocab_size = 10000  # Hanya 10.000 kata terpopuler
max_length = 200    # Maksimal 200 kata per review

(X_train_text, y_train_text), (X_test_text, y_test_text) = tf.keras.datasets.imdb.load_data(
    num_words=vocab_size
)

print(f"Training reviews: {len(X_train_text):,}")
print(f"Testing reviews : {len(X_test_text):,}")
print(f"Vocabulary size : {vocab_size:,} kata")
print(f"\nContoh review (dalam bentuk angka): {X_train_text[0][:20]}...")
print(f"Label: {'Positif' if y_train_text[0] == 1 else 'Negatif'}")

In [ ]:
# Decode review dari angka ke kata
word_index = tf.keras.datasets.imdb.get_word_index()
reverse_word_index = {v+3: k for k, v in word_index.items()}
reverse_word_index[0] = '<PAD>'
reverse_word_index[1] = '<START>'
reverse_word_index[2] = '<UNK>'

def decode_review(encoded):
    return ' '.join(reverse_word_index.get(i, '?') for i in encoded)

print("Contoh review:")
print("-" * 60)
print(decode_review(X_train_text[0])[:500] + "...")
print("-" * 60)
print(f"Sentimen: {'POSITIF' if y_train_text[0] == 1 else 'NEGATIF'}")

### Padding: Menyamakan Panjang Review

Setiap review punya panjang berbeda. Neural network butuh input ukuran sama, jadi kita lakukan **padding** — tambah angka 0 agar semua review punya panjang 200 kata.

```
Review pendek (80 kata) :  [4, 25, 7, ..., 12] + [0, 0, 0, ..., 0]  = 200 angka
Review panjang (350 kata): [4, 25, 7, ..., 99, ...] dipotong           = 200 angka
```

In [ ]:
# Padding — samakan panjang semua review menjadi 200 kata
X_train_pad = tf.keras.preprocessing.sequence.pad_sequences(X_train_text, maxlen=max_length, padding='post')
X_test_pad = tf.keras.preprocessing.sequence.pad_sequences(X_test_text, maxlen=max_length, padding='post')

print(f"Shape setelah padding: {X_train_pad.shape}")
print(f"Review terpendek menjadi 200 kata (ditambah 0)")
print(f"Review terpanjang dipotong menjadi 200 kata")

# Bangun model LSTM untuk analisis sentimen
model_sentiment = tf.keras.Sequential([
    tf.keras.layers.Embedding(vocab_size, 64, input_length=max_length, name='embedding'),
    tf.keras.layers.LSTM(64, name='lstm'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(32, activation='relu', name='dense'),
    tf.keras.layers.Dropout(0.5),
    tf.keras.layers.Dense(1, activation='sigmoid', name='output')
], name='LSTM_Sentiment')

model_sentiment.summary()

In [ ]:
model_sentiment.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print("Training LSTM Sentiment Analyzer...")
history_sent = model_sentiment.fit(
    X_train_pad, y_train_text,
    epochs=5,
    batch_size=128,
    validation_split=0.2,
    verbose=1
)
print("Selesai!")

# Evaluasi
test_loss, test_acc = model_sentiment.evaluate(X_test_pad, y_test_text, verbose=0)
print(f"\nAkurasi pada data test: {test_acc:.1%}")

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(history_sent.history['loss'], label='Training', linewidth=2)
ax1.plot(history_sent.history['val_loss'], label='Validation', linewidth=2)
ax1.set_title('Loss', fontweight='bold')
ax1.set_xlabel('Epoch')
ax1.legend()
ax1.grid(True, alpha=0.3)

ax2.plot(history_sent.history['accuracy'], label='Training', linewidth=2)
ax2.plot(history_sent.history['val_accuracy'], label='Validation', linewidth=2)
ax2.set_title('Accuracy', fontweight='bold')
ax2.set_xlabel('Epoch')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Training LSTM Sentiment Analyzer', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Demo prediksi beberapa review
print("\nDemo Prediksi Sentimen:")
print("=" * 60)
sample_indices = [0, 1, 2, 100, 200]
predictions = model_sentiment.predict(X_test_pad[sample_indices], verbose=0)

for idx, pred in zip(sample_indices, predictions):
    review_text = decode_review(X_test_text[idx])[:100]
    actual = "POSITIF" if y_test_text[idx] == 1 else "NEGATIF"
    predicted = "POSITIF" if pred[0] > 0.5 else "NEGATIF"
    confidence = pred[0] if pred[0] > 0.5 else 1 - pred[0]
    status = "[OK]" if actual == predicted else "[SALAH]"
    print(f"{status} Review: '{review_text}...'")
    print(f"   Aktual: {actual} | Prediksi: {predicted} ({confidence:.1%})")
    print()

## Bagian 5: Variasi Arsitektur RNN

RNN punya beberapa variasi arsitektur untuk tugas berbeda:

| Arsitektur | Input -> Output | Contoh Penggunaan |
|-----------|----------------|------------------|
| **One-to-One** | 1 -> 1 | Klasifikasi gambar (bukan RNN) |
| **One-to-Many** | 1 -> Urutan | Image captioning (gambar -> kalimat) |
| **Many-to-One** | Urutan -> 1 | Analisis sentimen (kalimat -> positif/negatif) |
| **Many-to-Many** | Urutan -> Urutan | Mesin terjemahan (kalimat -> kalimat) |

### GRU (Gated Recurrent Unit)

Selain LSTM, ada juga **GRU** — versi yang lebih sederhana dengan 2 gate (bukan 3). GRU sering kali sama bagusnya dengan LSTM tapi lebih cepat karena parameter lebih sedikit.

```
SimpleRNN  -->  2 gate (Forget, Input, Output)  -->  LSTM
SimpleRNN  -->  2 gate (Update, Reset)           -->  GRU
```

In [ ]:
# Bandingkan: SimpleRNN vs LSTM vs GRU pada prediksi sinus
model_gru = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(seq_length, 1)),
    tf.keras.layers.GRU(32, activation='tanh'),
    tf.keras.layers.Dense(1)
], name='GRU_Model')

model_gru.compile(optimizer='adam', loss='mse')
print("Training GRU...")
model_gru.fit(X_train, y_train, epochs=20, batch_size=32, verbose=0)
print("Selesai!")

y_pred_gru = model_gru.predict(X_test, verbose=0).flatten()
mse_gru = mean_squared_error(y_test, y_pred_gru)

# Perbandingan 3 model
print("\nPerbandingan Model Recurrent:")
print(f"{'Model':<12} {'Parameter':>10} {'MSE':>12}")
print("-" * 36)
for name, model, mse_val in [
    ('SimpleRNN', model_rnn, mse_rnn_val),
    ('LSTM', model_lstm, mse_lstm_val),
    ('GRU', model_gru, mse_gru)
]:
    print(f"{name:<12} {model.count_params():>10,} {mse_val:>12.6f}")

# Bar chart — skala MSE x 1.000.000 agar terlihat jelas
fig, ax = plt.subplots(figsize=(8, 5))
names = ['SimpleRNN', 'LSTM', 'GRU']
mses = [mse_rnn_val, mse_lstm_val, mse_gru]
mses_scaled = [m * 1e6 for m in mses]  # x 1.000.000
colors = ['#E53935', '#66BB6A', '#42A5F5']

bars = ax.bar(names, mses_scaled, color=colors, edgecolor='white', linewidth=2)
for bar, mse_val in zip(bars, mses):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.05,
            f'{mse_val:.6f}', ha='center', va='bottom', fontweight='bold', fontsize=11)
ax.set_ylabel('MSE (x 1.000.000) — lebih rendah = lebih baik')
ax.set_title('Perbandingan SimpleRNN vs LSTM vs GRU', fontsize=14, fontweight='bold')
ax.grid(axis='y', alpha=0.3)
ax.set_ylim(0, max(mses_scaled) * 1.4)  # beri ruang untuk label
plt.tight_layout()
plt.show()


## Bagian 6: Peran NVIDIA dalam RNN/LSTM

### Mengapa GPU NVIDIA Sangat Penting untuk RNN?

- **cuDNN LSTM/GRU**: NVIDIA menyediakan implementasi LSTM dan GRU yang dioptimasi khusus untuk GPU — hingga **6x lebih cepat** dari implementasi standar
- TensorFlow otomatis menggunakan cuDNN LSTM saat:
  - `activation='tanh'` (default)
  - `recurrent_activation='sigmoid'` (default)
  - GPU tersedia
- **NVIDIA NeMo**: Framework untuk membangun model NLP dan speech berbasis RNN/Transformer
- **Large Language Models** (GPT, dll) menggunakan Transformer (evolusi dari RNN+Attention) yang dilatih di ribuan GPU NVIDIA

> Fun fact: GPT-3 dilatih menggunakan sekitar 10.000 GPU NVIDIA V100 selama berminggu-minggu!

### Evolusi Arsitektur: RNN -> LSTM -> Transformer

```
RNN (1986)    -->  LSTM (1997)  -->  GRU (2014)  -->  Transformer (2017)  -->  GPT/BERT
Memori pendek     Memori panjang    Lebih simpel     Attention mechanism    State-of-the-art
```

In [ ]:
if tf.config.list_physical_devices('GPU'):
    print("cuDNN LSTM Status:")
    print(f"   CUDA available: {tf.test.is_built_with_cuda()}")
    print(f"   GPU: {tf.config.list_physical_devices('GPU')[0]}")
    print(f"\n   TensorFlow otomatis menggunakan cuDNN LSTM di GPU!")
    print(f"   cuDNN LSTM bisa 6x lebih cepat dari CPU implementation.")
else:
    print("Di Google Colab dengan GPU, LSTM otomatis menggunakan cuDNN NVIDIA")
    print("yang 6x lebih cepat dari implementasi CPU biasa.")
    print("\nAktifkan GPU: Runtime -> Change runtime type -> T4 GPU")

# Ringkasan model yang sudah dibuat
print("\n" + "=" * 50)
print("Ringkasan semua model yang dibuat:")
print("=" * 50)
models_summary = [
    ("SimpleRNN", model_rnn, "Prediksi sinus"),
    ("LSTM", model_lstm, "Prediksi sinus"),
    ("GRU", model_gru, "Prediksi sinus"),
    ("LSTM Sentiment", model_sentiment, "Analisis sentimen IMDB"),
]
print(f"{'Model':<20} {'Parameter':>10} {'Tugas':<30}")
print("-" * 62)
for name, model, task in models_summary:
    print(f"{name:<20} {model.count_params():>10,} {task:<30}")

## Kesimpulan

Selamat! Kamu telah menyelesaikan notebook tentang RNN dan LSTM. Berikut ringkasan yang sudah dipelajari:

---

### Yang Sudah Dipelajari

- Memahami data berurutan (sequential data) dan mengapa perlu RNN
- Cara kerja RNN: memproses input + memori dari langkah sebelumnya (hidden state)
- Masalah vanishing gradient pada RNN dan solusinya dengan LSTM
- Praktik prediksi gelombang sinus: SimpleRNN vs LSTM vs GRU
- Praktik analisis sentimen film IMDB dengan LSTM
- Variasi arsitektur RNN: One-to-One, Many-to-One, Many-to-Many
- Peran NVIDIA cuDNN dalam mempercepat training LSTM/GRU

---

### Kapan Menggunakan Apa?

| Situasi | Pilihan Model |
|---------|---------------|
| Data urutan pendek (< 10 langkah) | SimpleRNN |
| Data urutan panjang, teks | LSTM |
| Butuh keseimbangan kecepatan & akurasi | GRU |
| NLP modern, teks panjang | Transformer (BERT, GPT) |

---

### Selanjutnya

**Modul 06: NVIDIA GPU Acceleration untuk Deep Learning** — Pelajari bagaimana GPU NVIDIA, TensorRT, dan NeMo mempercepat training model deep learning secara dramatis.

---

*Navasena Generative ML Course — NCA-GENL*